## Wikipedia XML To SQLite
---

In [2]:
:dep xml-rs
:dep rusqlite

In [3]:
use std::fs::File;
use std::io::BufReader;
use xml::reader::{EventReader, XmlEvent};
use rusqlite::{Connection, Result as SqlResult};

In [4]:
#[derive(Debug)]
pub struct Page {
    pub title: String,
    pub text: String,
}

In [5]:
let home_path = std::env::var("HOME")?;
let file_name = "enwiki-latest-pages-articles-multistream.xml";
let file_path = home_path.clone() + "/work/wikipedia/" + file_name;
let file = File::open(file_path)?;
let reader = BufReader::new(file);
let parser = EventReader::new(reader);

In [6]:
let file_name = "wikipedia_dev.sqlite"
let file_path = home_path + "/work/wikipedia/" + file_name;
let conn = Connection::open(file_path)?;

In [7]:
conn.execute(
    "CREATE TABLE IF NOT EXISTS pages (
        id INTEGER PRIMARY KEY,
        title TEXT NOT NULL,
        text TEXT
    )",
    (),
)?;

In [8]:
let mut current_page: Option<Page> = None;
let mut in_title = false;
let mut in_text = false;

In [ ]:
{
    let mut stmt = conn.prepare("INSERT INTO pages (title, text) VALUES (?1, ?2)")?;
    for event in parser {
        match event? {
            XmlEvent::StartElement { name, .. } if name.local_name == "page" => {
                current_page = Some(Page {
                    title: String::new(),
                    text: String::new(),
                });
            }
            XmlEvent::StartElement { name, .. } if name.local_name == "title" => {
                in_title = true;
            }
            XmlEvent::StartElement { name, .. } if name.local_name == "text" => {
                in_text = true;
            }
            XmlEvent::Characters(data) => {
                if let Some(page) = current_page.as_mut() {
                    if in_title {
                        page.title.push_str(&data);
                    } else if in_text {
                        page.text.push_str(&data);
                    }
                }
            }
            XmlEvent::EndElement { name } if name.local_name == "title" => {
                in_title = false;
            }
            XmlEvent::EndElement { name } if name.local_name == "text" => {
                in_text = false;
            }
            XmlEvent::EndElement { name } if name.local_name == "page" => {
                if let Some(page) = current_page.take() {
                    stmt.execute(rusqlite::params![&page.title, &page.text])?;
                }
            }
            _ => {}
        }
    }
}